In [0]:
%sql
-- Q01 — Overall KPI summary: net revenue, distinct bookings, active passengers, average 
SELECT
    ROUND(SUM(annual_net_revenue), 2)                        AS net_revenue,
    SUM(distinct_booking)                                    AS distinct_bookings,
    SUM(active_passengers)                                    AS active_passengers,
    ROUND(SUM(annual_net_revenue) / SUM(distinct_booking), 2) AS average_booking_value
FROM global_airline.gold.vw_annual_booking_kpis

In [0]:
%sql
-- Q02 — Monthly revenue trend across all years (gold tables only)
SELECT
    date_format(f.fb_departuredate, 'yyyy-MM') AS revenue_month,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
WHERE f.fb_refundflag <> 'R'
GROUP BY 1
ORDER BY 1

In [0]:
%sql
-- Q03 — Annual revenue with year-over-year change
SELECT
    booking_year,
    annual_net_revenue,
    ROUND(annual_net_revenue - LAG(annual_net_revenue) OVER (ORDER BY booking_year), 2) AS yoy_change,
    ROUND(100 * (annual_net_revenue - LAG(annual_net_revenue) OVER (ORDER BY booking_year))
          / LAG(annual_net_revenue) OVER (ORDER BY booking_year), 2)                    AS yoy_change_pct
FROM global_airline.gold.vw_annual_booking_kpis
ORDER BY booking_year

In [0]:
%sql
-- Q04 — Revenue by passenger travel purpose (gold tables only)
SELECT
    p.pa_travelpurpose             AS travel_purpose,
    COUNT(DISTINCT f.fb_bookingid) AS bookings,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
JOIN global_airline.silver.booking b
  ON f.fb_bookingid = b.bk_bookingid
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
WHERE f.fb_refundflag <> 'R'
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q05 — Top 10 passengers by net revenue
SELECT
    p.pa_name,
    p.pa_travelpurpose,
    COUNT(DISTINCT b.bk_bookingid) AS bookings,
    ROUND(SUM(b.bk_totalprice), 2) AS net_revenue
FROM global_airline.silver.booking b
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
WHERE b.bk_status <> 'C'
GROUP BY p.pa_passengerid, p.pa_name, p.pa_travelpurpose
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q06 — Top 10 passengers as a share of total revenue
WITH per_passenger AS (
    SELECT bk_passengerid, SUM(bk_totalprice) AS revenue
    FROM global_airline.silver.booking
    WHERE bk_status <> 'C'
    GROUP BY 1
), totals AS (
    SELECT SUM(revenue) AS total_revenue FROM per_passenger
), top10 AS (
    SELECT SUM(revenue) AS top10_revenue
    FROM (SELECT revenue FROM per_passenger ORDER BY revenue DESC LIMIT 10)
)
SELECT
    ROUND(top10_revenue, 2) AS top10_revenue,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(100 * top10_revenue / total_revenue, 2) AS top10_share_pct
FROM top10 CROSS JOIN totals

In [0]:
%sql
-- Q07 — Top 10 routes by net revenue (flight segments)
SELECT
    r.ro_name       AS route,
    r.ro_routetype  AS route_type,
    COUNT(*)        AS segments,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
JOIN global_airline.gold.dim_route r
  ON f.fb_routeid = r.ro_routeid
GROUP BY 1, 2
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q08 — Bottom 10 routes by net revenue (flight segments)
SELECT
    r.ro_name      AS route,
    r.ro_routetype AS route_type,
    COUNT(*)       AS segments,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
JOIN global_airline.gold.dim_route r
  ON f.fb_routeid = r.ro_routeid
GROUP BY 1, 2
ORDER BY net_revenue ASC
LIMIT 10

In [0]:
%sql
-- Q09 — Net revenue by route type (domestic, short-haul, long-haul, ultra-long-haul)
SELECT
    r.ro_routetype AS route_type,
    COUNT(*)       AS segments,
    SUM(f.fb_seats) AS seats_sold,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
JOIN global_airline.gold.dim_route r
  ON f.fb_routeid = r.ro_routeid
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q10 — Net revenue by airline
SELECT
    a.al_name AS airline,
    COUNT(*)   AS segments,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
JOIN global_airline.gold.dim_airline a
  ON f.fb_airlineid = a.al_airlineid
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q11 — Airline revenue share (% of total)
WITH airline_revenue AS (
    SELECT a.al_name AS airline,
           SUM(f.fb_ticketprice * f.fb_seats) AS net_revenue
    FROM global_airline.gold.fact_bookings f
    JOIN global_airline.gold.dim_airline a
      ON f.fb_airlineid = a.al_airlineid
    GROUP BY 1
)
SELECT
    airline,
    ROUND(net_revenue, 2) AS net_revenue,
    ROUND(100 * net_revenue / SUM(net_revenue) OVER (), 2) AS revenue_share_pct
FROM airline_revenue
ORDER BY revenue_share_pct DESC

In [0]:
%sql
-- Q12 — Revenue by passenger region
SELECT
    g.r_name                    AS region,
    COUNT(DISTINCT b.bk_bookingid) AS bookings,
    ROUND(SUM(b.bk_totalprice), 2) AS net_revenue
FROM global_airline.silver.booking b
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
JOIN global_airline.gold.dim_geography g
  ON p.pa_nationkey = g.n_nationkey
WHERE b.bk_status <> 'C'
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q13 — Revenue by passenger nation (all nations)
SELECT
    g.n_name AS nation,
    g.r_name AS region,
    COUNT(DISTINCT b.bk_bookingid) AS bookings,
    ROUND(SUM(b.bk_totalprice), 2) AS net_revenue
FROM global_airline.silver.booking b
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
JOIN global_airline.gold.dim_geography g
  ON p.pa_nationkey = g.n_nationkey
WHERE b.bk_status <> 'C'
GROUP BY 1, 2
ORDER BY net_revenue DESC

In [0]:
%sql

-- Q14 — Revenue and volume split by refund flag
SELECT
    CASE fb_refundflag WHEN 'R' THEN 'Refunded' ELSE 'Non-refunded' END AS refund_status,
    COUNT(*)                                          AS segments,
    SUM(fb_seats)                                     AS seats_sold,
    ROUND(SUM(fb_ticketprice * fb_seats), 2)           AS net_revenue,
    ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS segment_share_pct
FROM global_airline.gold.fact_bookings
GROUP BY fb_refundflag
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q15 — Average discount by travel purpose
WITH booking_discount AS (
    SELECT fb_bookingid, AVG(fb_discount) AS avg_discount
    FROM global_airline.gold.fact_bookings
    GROUP BY 1
)
SELECT
    p.pa_travelpurpose             AS travel_purpose,
    COUNT(DISTINCT b.bk_bookingid) AS bookings,
    ROUND(AVG(bd.avg_discount), 2) AS avg_discount
FROM global_airline.silver.booking b
JOIN booking_discount bd
  ON b.bk_bookingid = bd.fb_bookingid
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
WHERE b.bk_status <> 'C'
GROUP BY 1
ORDER BY avg_discount DESC

In [0]:
%sql
-- Q16 — Average flight duration (departure to arrival, days) by cabin class
SELECT
    fb_cabinclass AS cabin_class,
    COUNT(*)      AS segments,
    ROUND(AVG(datediff(fb_arrivaldate, fb_departuredate)), 2) AS avg_flight_duration_days
FROM global_airline.gold.fact_bookings
GROUP BY 1
ORDER BY avg_flight_duration_days DESC

In [0]:
%sql
-- Q17 — Booking count and revenue by segment status (gold tables only)
-- (bk_bookingpriority is a silver-only column; fb_segmentstatus is the gold equivalent)
SELECT
    CASE fb_segmentstatus WHEN 'F' THEN 'Finished' WHEN 'O' THEN 'Open' END AS segment_status,
    COUNT(DISTINCT fb_bookingid)     AS bookings,
    ROUND(SUM(fb_ticketprice * fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q18 — Monthly breakdown of booking status
SELECT
    date_format(bk_bookingdate, 'yyyy-MM') AS revenue_month,
    bk_status                              AS booking_status,
    COUNT(DISTINCT bk_bookingid)           AS bookings,
    ROUND(SUM(bk_totalprice), 2)            AS net_revenue
FROM global_airline.silver.booking
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
-- Q19 — Route revenue ranking within each travel purpose
SELECT
    p.pa_travelpurpose             AS travel_purpose,
    r.ro_name                     AS route,
    r.ro_routetype                AS route_type,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue,
    RANK() OVER (
        PARTITION BY p.pa_travelpurpose
        ORDER BY SUM(f.fb_ticketprice * f.fb_seats) DESC
    ) AS revenue_rank
FROM global_airline.gold.fact_bookings f
JOIN global_airline.silver.booking b
  ON f.fb_bookingid = b.bk_bookingid
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
JOIN global_airline.gold.dim_route r
  ON f.fb_routeid = r.ro_routeid
WHERE f.fb_refundflag <> 'R'
GROUP BY 1, 2, 3
ORDER BY p.pa_travelpurpose, revenue_rank

In [0]:
%sql
-- Q20 — Revenue comparison: passenger geography vs airline geography by region
SELECT
    pg.r_name                             AS passenger_region,
    ag.r_name                             AS airline_region,
    COUNT(*)                              AS segments,
    ROUND(SUM(f.fb_ticketprice * f.fb_seats), 2) AS net_revenue
FROM global_airline.gold.fact_bookings f
JOIN global_airline.silver.booking b
  ON f.fb_bookingid = b.bk_bookingid
JOIN global_airline.gold.dim_passenger p
  ON b.bk_passengerid = p.pa_passengerid
JOIN global_airline.gold.dim_geography pg
  ON p.pa_nationkey = pg.n_nationkey
JOIN global_airline.gold.dim_airline a
  ON f.fb_airlineid = a.al_airlineid
JOIN global_airline.gold.dim_geography ag
  ON a.al_nationkey = ag.n_nationkey
WHERE f.fb_refundflag <> 'R'
GROUP BY 1, 2
ORDER BY net_revenue DESC

# Final Business Challenge — Booking Business Performance Summary

**Sources:** Bronze → Silver → Gold pipeline, Gold tables (`global_airline.gold.fact_bookings`, `dim_airline`, `dim_geography`, `dim_passenger`, `dim_route`) and `gold.vw_annual_booking_kpis`, the SQL queries Q01–Q20 above, plus the dashboard and Genie space built on the same Gold layer. All figures below come from executed query results. **This is synthetic practice data — findings describe this dataset only; no real-world causation is claimed.**

## A. Three Key Findings (directly supported by the data)

### A1. Stable ~$10–11M annual net-revenue run-rate, closing at the 5-year peak

* Q01 shows total net revenue of **$54,346,426.53** across **7,189** distinct bookings from **4,593** active passengers, at an average booking value of **$7,559.66**.
* Q03 shows annual net revenue peaked in 1998 at **$11,316,208.70** (+4.32% YoY); the only meaningful dip was 1995 at **$10,345,549.92** (−5.18%); every other year moved within ±5.7%.
* **Recommended management action:** Plan capacity and budgets on the demonstrated ~$10–11M annual run-rate; treat 1998's +4.32% as momentum to protect through schedule reliability, not a trend to extrapolate.

### A2. Long-haul flying produces the majority of segment revenue

* Q09 shows ultra-long-haul (**$31,673,277.56**) + long-haul (**$30,248,759.13**) = **$61,922,036.69**, i.e. **54.4%** of the $113,762,323.08 gross segment revenue, from **13,622 of 25,000** segments (54.5%).
* Per-seat revenue is nearly flat across route types (Q09: ≈ **$1,803–$1,830 per seat**), so the concentration is driven by volume, not pricing.
* **Recommended management action:** Prioritize capacity, uptime, and operational attention on long-haul and ultra-long-haul routes, where revenue concentrates.

### A3. Revenue is highly diversified — no concentration exposure

* Q11 shows the largest airline (AmeriJet) holds only a **10.18%** revenue share; Q10 shows the top 10 airlines cluster tightly between **$11.21M and $11.58M**.
* Q06 shows the top 10 passengers contribute only **$1,024,952.58 (1.89%)** of total revenue; Q05 shows the single largest passenger (Mary Williams) at **$118,578.71**.
* Q12 shows even regional spread (**$10.03M–$11.47M** across all five regions); Q13 shows the largest nation (France) at **$2,624,149.76**.
* **Recommended management action:** Pursue broad-based, network-wide demand growth; single-account or single-partner wins cannot materially move total revenue.

## B. Two Opportunities (observed data + clearly-labeled hypothesis)

### B1. Premium-cabin mix is thin (observed) → upsell headroom (hypothesis)

* **Data:** Q16 shows Economy = **15,115 of 25,000 segments (60.5%)**, while First = **1,249 (5.0%)** and Business = **3,692 (14.8%)**.
* **Hypothesis (not measured in the data):** converting even a small share of Economy segments to premium cabins would lift average revenue per segment. The dataset shows only the current mix — no upgrade-conversion or price-elasticity data exists.
* **Recommended management action:** Pilot premium upsell offers on long-haul routes (Q09's highest-revenue category) and measure conversion before scaling.

### B2. Discounts are uniform across all customer segments (observed) → segmentation headroom (hypothesis)

* **Data:** Q15 shows an identical average discount of **0.07** across all five travel purposes; Q04 shows revenue nearly even across purposes (**$19.47M–$21.91M**).
* **Hypothesis (not measured in the data):** purpose-based discount differentiation could protect margin on price-insensitive segments. The data contains no price-response signal to confirm this.
* **Recommended management action:** Run a controlled discount test by travel purpose and track booking-volume response.

## C. Two Risks / Areas Requiring Further Investigation (directly supported by the data)

### C1. ~25% of bookings are still “Open” and ~10% of segments are refunded

* Q17 shows **1,896 of 7,621 bookings (24.9%)** in Open status, carrying **$28,687,079.73** (25.2% of the $113.76M gross segment revenue) that is not yet finalized.
* Q14 shows **2,504 segments (10.02%)** flagged Refunded, representing **$11,421,547.39** of gross ticket value; Q18 shows cancellations recur every month (e.g., **15** cancelled bookings in 1994-01).
* **Recommended management action:** Investigate the aging and root causes of Open segments (operational backlog vs. data-quality artifact) and quantify refund/cancellation drivers before counting Open-segment revenue as durable.

### C2. Inconsistent metric definitions and data-granularity gaps in the Gold layer

* The same “net revenue” label yields materially different totals: Q01 (vw_annual_booking_kpis) = **$54.35M**; Q04 (booking-level, refund-excluded) = **$102.34M**; Q09/Q14/Q17 (gross segment-level) = **$113.76M**.
* Q14 also reveals two distinct non-refund codes (A, N) collapsed under one “Non-refunded” label; Q16 shows average flight duration of **0 days** for every cabin class (arrival dates equal departure dates at day granularity); and passenger-level analyses must join `silver.booking` (Q04, Q05, Q12, Q13, Q18) because Gold lacks a passenger–booking link.
* **Recommended management action:** Standardize one canonical revenue metric (net of refunds, booking grain) in the Gold layer, document the view's definition, add the passenger link to Gold, and capture finer-grained timestamps before any duration or lead-time analysis.

---

**Data vs. assumptions:** Findings A1–A3 and risks C1–C2 are directly observed in executed query results. Opportunities B1–B2 combine observed data (cabin mix, discount uniformity) with explicitly-labeled hypotheses about revenue potential. All numbers come from synthetic practice data and describe this dataset only.